# Unidad 11 · Modelos lineales generalizados: logística y Poisson

Reproduce el ejemplo resuelto (Reparto Biobío y la imprenta de etiquetas) y la pieza del caso Estampa Sur. Lee los CSV de `../datos/`. Al final, el diccionario `RES` junta las cifras que el verificador compara.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
DATOS = '../datos/'

## Ejemplo A · pedidos atrasados (logística binomial agrupada)

Primero solo la lluvia, a mano desde la tabla 2 × 2; después el modelo con distancia.

In [ ]:
rep = pd.read_csv(DATOS + 'u11_ejemplo_reparto.csv')
rep['a_tiempo'] = rep.pedidos - rep.atrasados
t = rep.groupby('lluvia')[['atrasados', 'a_tiempo']].sum()
print(t)
odds = t.atrasados / t.a_tiempo
beta1_mano = np.log(odds[1] / odds[0])
se1_mano = np.sqrt((1 / t.values).sum())
print('beta1 =', round(beta1_mano, 4), ' EE =', round(se1_mano, 4), ' OR =', round(np.exp(beta1_mano), 3))

In [ ]:
g1 = smf.glm('atrasados + a_tiempo ~ lluvia', data=rep, family=sm.families.Binomial()).fit()
g2 = smf.glm('atrasados + a_tiempo ~ km + lluvia', data=rep, family=sm.families.Binomial()).fit()
print(g2.summary())
print('odds ratio:', np.exp(g2.params).round(3).to_dict())
print('devianza residual', round(g2.deviance, 2), 'con', int(g2.df_resid), 'gl')

In [ ]:
nuevos = pd.DataFrame({'km': [2, 2, 6, 6, 10, 10], 'lluvia': [0, 1] * 3})
nuevos['p_atraso'] = g2.predict(nuevos)
nuevos

## Ejemplo B · atascos por turno (Poisson)

In [ ]:
at = pd.read_csv(DATOS + 'u11_ejemplo_atascos.csv')
print(at.groupby('reciclado').atascos.agg(['sum', 'mean', 'var']))
gp = smf.glm('atascos ~ reciclado', data=at, family=sm.families.Poisson()).fit()
print(gp.summary())
phi = gp.pearson_chi2 / gp.df_resid
print('razón de tasas', round(np.exp(gp.params['reciclado']), 3), ' dispersión', round(phi, 2))

## Caso · Estampa Sur

Logística del defecto según temperatura y tinta al agua; Poisson de los reclamos diarios según la campaña web.

In [ ]:
lotes = pd.read_csv(DATOS + 'lotes.csv')
print(pd.crosstab(lotes.tinta_agua, lotes.defecto, margins=True))
m4 = smf.logit('defecto ~ temperatura + tinta_agua', data=lotes).fit(disp=0)
print(m4.summary())
print('odds ratio tinta al agua:', round(np.exp(m4.params['tinta_agua']), 2))
print('razón de verosimilitud G =', round(m4.llr, 2), ' p =', m4.llr_pvalue)

In [ ]:
pred = (m4.predict() >= 0.5).astype(int)
print(pd.crosstab(lotes.defecto, pred, rownames=['real'], colnames=['predicho']))

In [ ]:
rec = pd.read_csv(DATOS + 'reclamos_dia.csv')
print(rec.groupby('campana_web').reclamos.agg(['sum', 'count', 'mean', 'var']))
p5 = smf.glm('reclamos ~ campana_web', data=rec, family=sm.families.Poisson()).fit()
print(p5.summary())
phi_c = p5.pearson_chi2 / p5.df_resid
print('razón de tasas', round(np.exp(p5.params['campana_web']), 2), ' dispersión', round(phi_c, 2))

In [ ]:
RES = {'ej_beta_lluvia_2x2': beta1_mano, 'ej_se_lluvia_2x2': se1_mano,
       'ej_beta_km': g2.params['km'], 'ej_beta_lluvia': g2.params['lluvia'], 'ej_dev': g2.deviance,
       'ej_rr_atascos': np.exp(gp.params['reciclado']), 'ej_phi_atascos': phi,
       'caso_or_tinta': np.exp(m4.params['tinta_agua']), 'caso_beta_temp': m4.params['temperatura'],
       'caso_G': m4.llr, 'caso_rr_web': np.exp(p5.params['campana_web']), 'caso_phi': phi_c}
RES